In [59]:
import pandas as pd
import numpy as np

df_near = pd.read_csv("data/near_earth_asteroids_2025.csv")
df_near.head()

df_close = pd.read_csv("data/asteroid_close_approaches_2015_2035.csv")
df_close.head()

df_close.info()
df_close['close_approach_date'] = pd.to_datetime(df_close['close_approach_date'], format="ISO8601")
df_close['close_approach_date']

df_close['is_future'].value_counts(normalize=True)

<class 'pandas.DataFrame'>
RangeIndex: 27430 entries, 0 to 27429
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   designation             27430 non-null  str    
 1   full_name               27430 non-null  str    
 2   close_approach_date     27430 non-null  str    
 3   distance_au             27430 non-null  float64
 4   dist_km                 27430 non-null  float64
 5   dist_lunar              27430 non-null  float64
 6   distance_min_au         27430 non-null  float64
 7   distance_max_au         27430 non-null  float64
 8   velocity_km_s           27430 non-null  float64
 9   v_rel_kmh               27430 non-null  float64
 10  velocity_infinity_km_s  27413 non-null  float64
 11  absolute_magnitude      27422 non-null  float64
 12  is_future               27430 non-null  bool   
dtypes: bool(1), float64(9), str(3)
memory usage: 3.6 MB


/tmp/ipykernel_1022688/2650471278.py:4: DtypeWarning: Columns (0: name) have mixed types. Specify dtype option on import or set low_memory=False.
  df_near = pd.read_csv("data/near_earth_asteroids_2025.csv")


is_future
False    0.858257
True     0.141743
Name: proportion, dtype: float64

In [60]:
## We can use all rows where 'is_future' column is True as our inference 

df_near['first_obs'] = pd.to_datetime(df_near['first_obs'], format="ISO8601")
df_near['first_obs']

df_near['last_obs'] = pd.to_datetime(df_near['last_obs'], format="ISO8601")
df_near['last_obs']

df_near.info()

print(df_near.iloc[: , 0:5])
print(df_close[['designation','full_name']].head(30))

<class 'pandas.DataFrame'>
RangeIndex: 41281 entries, 0 to 41280
Data columns (total 29 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   spkid                  41281 non-null  int64         
 1   full_name              41281 non-null  str           
 2   pdes                   41281 non-null  str           
 3   name                   182 non-null    str           
 4   pha                    41281 non-null  bool          
 5   H                      41279 non-null  float64       
 6   diameter_km            41281 non-null  float64       
 7   diameter_m             41281 non-null  float64       
 8   diameter_is_estimated  41281 non-null  bool          
 9   size_category          41281 non-null  str           
 10  albedo                 1204 non-null   float64       
 11  rot_per                2181 non-null   float64       
 12  class                  41281 non-null  str           
 13  e           

In [61]:
df_close[df_close['full_name'].duplicated()]
df_close[df_close['designation'] == "2022 BY39"]
df_close['full_name'].nunique()

df_near['full_name'].nunique() == len(df_near)
df_near[df_near['last_obs'] < "2015-01-01"] ## all asteroids that were last seen before 2015. We can leave these out


df_close[df_close['designation'] == "2022 AP1"]
df_near['full_name'] = df_near['full_name'].apply(lambda x: x[x.index('('): ])





In [62]:
# df_near
# #df_close['full_name'].apply(lambda x: x[x.index('(')+1: -1])
# part =  df_close[df_close['full_name'].apply(lambda x: x.index(" ")) > 0]['full_name'].apply(lambda x: x[x.index('(')+1: -1])
# part_idx = df_close[df_close['full_name'].apply(lambda x: x.index(" ")) > 0]['full_name'].str.partition(" ").iloc[: , -1].index

# part_idx # length=864
# full = df_close[df_close['full_name'].apply(lambda x: x.index(" ") == 0)]['full_name'] # Length: 26566 
# full_idx = full.index


# (26566 + 864) == len(df_close) # full and part cover all values and indexes in the column 'full_name'
# part

# full_idx

# for idx, val in part.items():
#     df_close.loc[idx, ["full_name"] ] = val



In [63]:
# df_close['full_name'].apply(lambda x: x[x.index(' (') + 1: -1])
# for k, v in part.items():
#     print(f"{k}: {v}")



# df_close['full_name'].apply(lambda x: x[x.index(" ")])
part =  df_close[df_close['full_name'].apply(lambda x: x.index(" ")) > 0]['full_name'].apply(lambda x: x[x.index('('):x.index(')')+1])
part

for k, v in part.items():
    if "(" in v:
      df_close.loc[k, ['full_name']] = v



df_close[df_close['full_name'].apply(lambda x: x.index("")) > 0]['full_name']


#df_close[df_close['full_name'].apply(lambda x: '(' not in x)]['full_name']





Series([], Name: full_name, dtype: str)

In [64]:
idx = df_close[df_close['full_name'].apply(lambda x: '(' not in x)]['full_name'].index

df_temp = df_close.drop(idx, axis=0)
df_temp['full_name'].apply(lambda x: x[x.index('(')+1:-1])

## The code in this cell verifies that the column 'full_name' only has values enclosed in parentheses when indexes idx are dropped




0         2022 AP1
1        2015 AE45
2        2005 YQ96
3        2014 YQ34
4        2014 YE42
           ...    
27425     2025 AC1
27426     2024 YF1
27427     2025 PN7
27428     2022 YS6
27429    2024 WX15
Name: full_name, Length: 27422, dtype: str

In [65]:
## Now we can finally join df_near and df_close on the column 'full_name'

df_near['full_name'] = df_near['full_name'].str.strip()
df_close['full_name'] = df_close['full_name'].str.strip()

near_set = set(df_near['full_name'].tolist()) 
close_set = set(df_close['full_name'].tolist())

close_set.difference(near_set)

{'10145 (1994 CK1)',
 '10302 (1989 ML)',
 '11500 Tomaiyowit (1989 UR)',
 '14827 Hypnos (1986 JK)',
 '1566 Icarus (1949 MA)',
 '1917 Cuyo (1968 AA)',
 '1981 Midas (1973 EA)',
 '20236 (1998 BZ7)',
 '2061 Anza (1960 UA)',
 '2329 Orthos (1976 WA)',
 '249P/LINEAR',
 '25143 Itokawa (1998 SF36)',
 '252P/LINEAR',
 '27002 (1998 DV9)',
 '289P/Blanpain',
 '29075 (1950 DA)',
 '3122 Florence (1981 ET3)',
 '3200 Phaethon (1983 TB)',
 '33342 (1998 WT24)',
 '3361 Orpheus (1982 HR)',
 '35107 (1991 VH)',
 '35396 (1997 XF11)',
 '37638 (1993 VB)',
 '41429 (2000 GE2)',
 '4450 Pan (1987 SY)',
 '4581 Asclepius (1989 FC)',
 '45P/Honda-Mrkos-Pajdusakova',
 '460P/PANSTARRS',
 '4660 Nereus (1982 DB)',
 '46P/Wirtanen',
 '4953 (1990 MU)',
 '5189 (1990 UQ)',
 '52768 (1998 OR2)',
 '5604 (1992 FE)',
 '5645 (1990 SP)',
 '5693 (1993 EA)',
 '5731 Zeus (1988 VP4)',
 '6037 (1988 EG)',
 '6063 Jason (1984 KB)',
 '6239 Minos (1989 QF)',
 '65690 (1991 DG)',
 '65717 (1993 BX3)',
 '65803 Didymos (1996 GT)',
 '66008 (1998 QH2)',

In [66]:
df_near.info()
X_near = df_near.drop(['spkid','full_name','pdes','name'], axis=1)
X_near['class'].value_counts()
#X_near.info()
X_near[['e', 'a', 'i', 'q']].describe()

<class 'pandas.DataFrame'>
RangeIndex: 41281 entries, 0 to 41280
Data columns (total 29 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   spkid                  41281 non-null  int64         
 1   full_name              41281 non-null  str           
 2   pdes                   41281 non-null  str           
 3   name                   182 non-null    str           
 4   pha                    41281 non-null  bool          
 5   H                      41279 non-null  float64       
 6   diameter_km            41281 non-null  float64       
 7   diameter_m             41281 non-null  float64       
 8   diameter_is_estimated  41281 non-null  bool          
 9   size_category          41281 non-null  str           
 10  albedo                 1204 non-null   float64       
 11  rot_per                2181 non-null   float64       
 12  class                  41281 non-null  str           
 13  e           

,e,a,i,q
count,41281.000000,41281.000000,41281.000000,41281.000000
mean,0.434323,1.757387,11.834802,0.914116
std,0.176975,2.119936,10.549392,0.221873
min,0.002800,0.461800,0.010000,0.069000
25%,0.300300,1.287000,4.350000,0.792000
50%,0.448200,1.679000,8.400000,0.962000
75%,0.563000,2.164000,16.550000,1.057000
max,0.996400,350.300000,165.600000,1.300000


In [67]:
X_near = df_near.drop(['spkid','full_name','pdes','name', 'diameter_m','moid_lunar_distances',\
                       'moid_km', 'per_y', 'data_arc_years', 'diameter_is_estimated','pha', 'first_obs', 'last_obs'], axis=1)
#X_near[['per', 'per_y']]
# per --> how many Earth days the asteroid takes to complete one full revolution
# per_y --> how many Earth years the asteroid takes to complete one full revolution

X_near.rename({'q':'perihelion_distance_au', 'ad':'aphelion_distance_au', 'e':'eccentricity', 'a':'semimajor_axis_au'\
               , 'i':'inclination_deg','per':'orbital_period_days','n':'mean_motion_deg_day', 'rot_per':'rot_per_h',\
                'class':'class_code'}, axis=1, inplace=True)
X_near.info()


<class 'pandas.DataFrame'>
RangeIndex: 41281 entries, 0 to 41280
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   H                       41279 non-null  float64
 1   diameter_km             41281 non-null  float64
 2   size_category           41281 non-null  str    
 3   albedo                  1204 non-null   float64
 4   rot_per_h               2181 non-null   float64
 5   class_code              41281 non-null  str    
 6   eccentricity            41281 non-null  float64
 7   semimajor_axis_au       41281 non-null  float64
 8   inclination_deg         41281 non-null  float64
 9   perihelion_distance_au  41281 non-null  float64
 10  aphelion_distance_au    41281 non-null  float64
 11  orbital_period_days     41281 non-null  float64
 12  moid_au                 41150 non-null  float64
 13  mean_motion_deg_day     41281 non-null  float64
 14  condition_code          41279 non-null  float64
 

In [68]:
# X_near['pha'].value_counts() # The target column pha (Potentially Hazardous Asteroid) is imbalanced, with 
#                              # True valued labels being only around 6% of labels
                            
# len(X_near[X_near['pha']==True]) / len(X_near['pha']) # .067

# X_near['pha'].value_counts() # 2539 True values

X_near.info()

# X_near[['moid_lunar_distances', 'moid_km']]
# X_near[['data_arc', 'data_arc_years']]

#X_near2 = X_near.drop(['moid_lunar_distance','data_arc_years', 'per_y', 'data_arc_years'], axis=1)


#X_near['size_category'] = X_near['size_category'].apply(lambda x: x[0:x.index(" ")].strip())


cols_with_nan = [col for col, val in X_near.isna().sum().items() if val > 0]
cols_with_nan

numeric_columns = X_near.select_dtypes(include='number').columns.tolist()
numeric_columns

string_columns = X_near.select_dtypes(include='str').columns.tolist()
string_columns

cols_with_nan
df_near['condition_code']





<class 'pandas.DataFrame'>
RangeIndex: 41281 entries, 0 to 41280
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   H                       41279 non-null  float64
 1   diameter_km             41281 non-null  float64
 2   size_category           41281 non-null  str    
 3   albedo                  1204 non-null   float64
 4   rot_per_h               2181 non-null   float64
 5   class_code              41281 non-null  str    
 6   eccentricity            41281 non-null  float64
 7   semimajor_axis_au       41281 non-null  float64
 8   inclination_deg         41281 non-null  float64
 9   perihelion_distance_au  41281 non-null  float64
 10  aphelion_distance_au    41281 non-null  float64
 11  orbital_period_days     41281 non-null  float64
 12  moid_au                 41150 non-null  float64
 13  mean_motion_deg_day     41281 non-null  float64
 14  condition_code          41279 non-null  float64
 

0        0.0
1        0.0
2        0.0
3        0.0
4        0.0
        ... 
41276    9.0
41277    9.0
41278    9.0
41279    8.0
41280    0.0
Name: condition_code, Length: 41281, dtype: float64

In [69]:
X_near.columns.tolist()

['H',
 'diameter_km',
 'size_category',
 'albedo',
 'rot_per_h',
 'class_code',
 'eccentricity',
 'semimajor_axis_au',
 'inclination_deg',
 'perihelion_distance_au',
 'aphelion_distance_au',
 'orbital_period_days',
 'moid_au',
 'mean_motion_deg_day',
 'condition_code',
 'data_arc']

In [70]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
import numpy as np

size_mapping = {'Large':3, 'Medium':2, 'Small':1, 'Tiny':0}
y = df_near['pha']



# preprocessor = ColumnTransformer(
#     transformers=[
#         ('ohe', OneHotEncoder(handle_unknown='ignore', drop='first'),['class', 'size_category'] ),
#         ('simple_imputer', SimpleImputer(missing_values=np.nan, strategy="mean"), cols_with_nan),
      
#     ],
#     remainder='passthrough'
# )

categorical_transformer = Pipeline(
    steps = [
        ("one_hot_encoder", OneHotEncoder(handle_unknown='ignore', drop='first'))
    ]
)

numeric_transformer = Pipeline(
    steps = [
        ('simple_imputer', SimpleImputer()),
        ('standard_scaler', StandardScaler())
    ]
)

preprocessor = ColumnTransformer(
    transformers = [
        ('categorical_transformer', categorical_transformer, string_columns),
        ('numeric_transformer', numeric_transformer, numeric_columns)
    ]
)

# pipeline_lr= Pipeline( steps= [
#     ('preprocessor', preprocessor),
#     ('standard_scaler', StandardScaler(), numeric_columns)
#     ('logistic_regression', LogisticRegression())
# ]
# )

# pipeline_rf = Pipeline( steps= [
#     ('preprocessor', preprocessor),
#     ('random_forest', RandomForestClassifier())
# ])

pipeline_lr = make_pipeline(preprocessor, LogisticRegression())
pipeline_rf = make_pipeline(preprocessor, RandomForestClassifier())
pipeline_gbc = make_pipeline(preprocessor, GradientBoostingClassifier())



pipelines = {
    "Random Forest": pipeline_rf,

    "Logistic Regression": pipeline_lr,

    "Gradient Boosting" : pipeline_gbc
}


X_train, X_test, y_train, y_test = train_test_split(X_near, y, test_size=0.2, random_state=42)

stratified_kf = StratifiedKFold(n_splits=5, shuffle=False)


for name, pipeline in pipelines.items():
    scores = cross_val_score(pipeline, X_train, y_train, cv=stratified_kf, scoring='f1_micro')

    print(f"{name:20s}: f1 score = {np.mean(scores):.3f} ± {np.std(scores):.3f}")

# scores_lr = cross_val_score(pipeline_lr, X_train, y_train, cv=stratified_kf, scoring='f1_micro')
# scores_rf = cross_val_score(pipeline_rf, X_train, y_train, cv=stratified_kf, scoring='f1_micro')




Random Forest       : f1 score = 0.998 ± 0.001
Logistic Regression : f1 score = 0.993 ± 0.001
Gradient Boosting   : f1 score = 0.999 ± 0.000


In [71]:
rf_param_grid = {
    "randomforestclassifier__n_estimators": [10, 70, 100, 500],
    "randomforestclassifier__max_depth": [2, 5, 7, 10, None],
    "randomforestclassifier__min_samples_leaf": [2, 3, 4]
}

rf_grid = GridSearchCV(
    pipelines["Random Forest"],
    rf_param_grid,
    cv=5,
    scoring='f1',
    n_jobs = -1,
    return_train_score=True
)

pipelines["Random Forest"].get_params().keys()

rf_grid.fit(X_train, y_train)
best_f1 = rf_grid.best_score_

print(f"Best parameters: {rf_grid.best_params_}")
print(f"\nBest score: {best_f1}")



Best parameters: {'randomforestclassifier__max_depth': 10, 'randomforestclassifier__min_samples_leaf': 3, 'randomforestclassifier__n_estimators': 100}

Best score: 0.9893037224522565


In [72]:
import joblib
import dill

best_model = rf_grid.best_estimator_
rf_grid_predict = best_model.predict(X_test)

print(f1_score(rf_grid_predict, y_test)) #0.9857549857549858

best_model['randomforestclassifier']

0.9848484848484849


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",3
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [73]:
with open('./artifacts/best_model.pkl', 'wb') as file:
    joblib.dump(best_model['randomforestclassifier'], file)

best_model.get_params().keys()
best_model['randomforestclassifier']

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",3
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [74]:
from sklearn.metrics import f1_score
from sklearn.model_selection import KFold


#scores_rf = cross_val_score(pipeline_rf, X_train, y_train, cv=stratified_kf, scoring='f1_macro')

# print(scores_lr)
# print(scores_rf)

pipeline_lr.fit(X_train,y_train)
y_predict_lr = pipeline_lr.predict(X_test)

pipeline_rf.fit(X_train, y_train)
y_predict_rf = pipeline_rf.predict(X_test)

print(f1_score(y_test, y_predict_lr))
print(f1_score(y_test, y_predict_rf))

pipeline_lr


0.9337175792507204
0.9867172675521821


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('columntransformer', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical_transformer', ...), ('numeric_transformer', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, def

In [75]:
from sklearn.model_selection import KFold

skf = StratifiedKFold(n_splits=3)

for train, test in skf.split(X_near, y):
    print(np.bincount(y[train]))
    print(np.bincount(y[test]))

    print("==========")


kf = KFold(n_splits=3)
print("\n \n")
for train, test in kf.split(X_near, y):
    print(y[test])
    #print(np.bincount(y[test]))

    print("==========")

[25828  1692]
[12914   847]
[25828  1693]
[12914   846]
[25828  1693]
[12914   846]

 

0        False
1        False
2        False
3        False
4        False
         ...  
13756    False
13757    False
13758    False
13759    False
13760    False
Name: pha, Length: 13761, dtype: bool
13761    False
13762    False
13763    False
13764    False
13765    False
         ...  
27516    False
27517    False
27518    False
27519    False
27520    False
Name: pha, Length: 13760, dtype: bool
27521    False
27522    False
27523     True
27524    False
27525    False
         ...  
41276    False
41277    False
41278    False
41279    False
41280     True
Name: pha, Length: 13760, dtype: bool
